In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.optimize import minimize
import polars as pl
import plotly.express as px

In [ ]:
def getLoss(pointsXPositions, pointsYPositions):
    def loss(x, y, r):
        distVector = np.sqrt((pointsXPositions - x) ** 2 + (pointsYPositions - y) ** 2)
        return np.sum(((distVector - r)**2))
    return loss

In [ ]:
from math import dist
def getGradient(pointsXPositions, pointsYPositions):
    def gradientLoss(x, y, r):
        distVector = np.sqrt((pointsXPositions - x) ** 2 + (pointsYPositions - y) ** 2)
        xComponent = 2 * np.sum((x - pointsXPositions) * (distVector - r) / distVector)
        yComponent = 2 * np.sum((y - pointsYPositions) * (distVector - r) / distVector)
        rComponent = np.sum(2 * (r - distVector))
        return xComponent, yComponent, rComponent
    return gradientLoss

In [ ]:
from numpy.matlib import gradient
nPoints = 3
lim = 1
xlim = (-lim, lim)
ylim = (-lim, lim)
pointsXPositions = np.random.rand(nPoints)
pointsYPositions = np.random.rand(nPoints)

nIterations = 1000

centerX = 0.0
centerY = 0.0
radius = 1.0

lossFunction = getLoss(pointsXPositions, pointsYPositions)
gradientFunction = getGradient(pointsXPositions, pointsYPositions)

print(lossFunction(centerX, centerY, radius))



df = pl.DataFrame({
    'id': [0],
    'centerX': [centerX],
    'centerY': [centerY],
    'radius': [radius],
    'loss': [lossFunction(centerX, centerY, radius)],
    'gradientX': [gradientFunction(centerX, centerY, radius)[0]],
    'gradientY': [gradientFunction(centerX, centerY, radius)[1]],
    'gradientR': [gradientFunction(centerX, centerY, radius)[2]],
})

def updateDataFrameCallback(params):
    centerX, centerY, radius = params
    loss = lossFunction(centerX, centerY, radius)
    global df
    current_id = df['id'].max() + 1
    new_row = pl.DataFrame({
        'id': [current_id],
        'centerX': [centerX],
        'centerY': [centerY],
        'radius': [radius],
        'loss': [loss],
        'gradientX': [gradientFunction(centerX, centerY, radius)[0]],
        'gradientY': [gradientFunction(centerX, centerY, radius)[1]],
        'gradientR': [gradientFunction(centerX, centerY, radius)[2]],
    })
    

    df = df.vstack(new_row)

res = minimize(
    lambda params: lossFunction(params[0], params[1], params[2]), 
    [centerX, centerY, radius],
    #bounds = [(None, None), (None, None), (0, None)],
    method='L-BFGS-B', 
    #method='Nelder-Mead',
    options={'maxiter': nIterations},
    jac=lambda params: gradientFunction(params[0], params[1], params[2]),
    callback=updateDataFrameCallback
    )
centerX, centerY, radius = res.x

thetas = pl.DataFrame({'theta': np.linspace(0, 2 * np.pi, 1000)})
df = df.join(thetas, how='cross')
df = df.with_columns(
    x=pl.col('centerX') + pl.col('radius') * pl.col('theta').cos(),
    y=pl.col('centerY') + pl.col('radius') * pl.col('theta').sin(),
)


print(lossFunction(centerX, centerY, radius))
fig = px.line(df, x='x', y='y', animation_frame='id', title='Circle fitting')
fig.add_trace(go.Scatter(x=pointsXPositions, y=pointsYPositions, mode='markers', name='Points'))
fig.update_layout(xaxis_range=xlim, yaxis_range=ylim)
fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.show()

In [ ]:
dfGrouped = df.group_by('id').agg(
    [pl.col('loss').first(), 
    pl.col("radius").first(), 
    pl.col("centerX").first(), 
    pl.col("centerY").first(), 
    pl.col("gradientX").first(), 
    pl.col("gradientY").first(),
    pl.col("gradientR").first()])
dfGrouped

In [ ]:
fig = px.line(dfGrouped, x='id', y='loss', title='Loss over iterations')
fig.show()

In [ ]:
fig = px.line(dfGrouped, x='id', y='radius', title='Radius over iterations')
fig.show()

In [ ]:
fig = px.line(dfGrouped, x='id', y='centerX', title='Radius over iterations')
fig.show()

In [ ]:
fig = px.line(dfGrouped, x='id', y='centerY', title='Radius over iterations')
fig.show()

In [ ]:
lossFunction(centerX, centerY, radius)


In [ ]:
np.sum(((pointsXPositions - centerX)**2 + (pointsYPositions - centerY)**2 - radius**2)**2)

In [ ]:
df

In [ ]:
np.sum(np.sqrt((pointsXPositions - centerX) ** 2 + (pointsYPositions - centerY) ** 2) - radius) ** 2

In [ ]:
radius

In [ ]:
np.sqrt((pointsXPositions - centerX) ** 2 + (pointsYPositions - centerY) ** 2) - radius

In [ ]:
radius